<a href="https://colab.research.google.com/github/J3rmed/Estructura-de-Base-de-Datos/blob/main/Sistema_de_B%C3%BAsqueda_ABB_B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import random
import time
import sys
sys.setrecursionlimit(20000) # Necesario para el ABB degenerado

# ==========================================
# 1. GENERACIÓN DE DATOS
# ==========================================
def generar_estudiantes(cantidad=10000, ordenado=False):
    estudiantes = []
    ids = list(range(1000, 1000 + cantidad))
    if not ordenado:
        random.shuffle(ids)

    nombres = ["Ana", "Carlos", "María", "Juan", "Luis", "Elena", "Pedro", "Lucía"]
    for i in ids:
        estudiantes.append({
            "id": i,
            "nombre": random.choice(nombres),
            "promedio": round(random.uniform(5.0, 10.0), 1)
        })
    return estudiantes

# ==========================================
# 2. ESTRUCTURA 1: LISTA (Búsqueda Lineal)
# ==========================================
class BusquedaLista:
    def __init__(self):
        self.datos = []

    def insertar(self, estudiante):
        self.datos.append(estudiante)

    def buscar(self, id_buscado):
        for est in self.datos:
            if est["id"] == id_buscado:
                return est
        return None

# ==========================================
# 3. ESTRUCTURA 2: ÁRBOL ABB
# ==========================================
class NodoABB:
    def __init__(self, estudiante):
        self.estudiante = estudiante
        self.izq = None # Valores menores
        self.der = None # Valores mayores

class ArbolABB:
    def __init__(self):
        self.raiz = None

    def insertar(self, estudiante):
        if self.raiz is None:
            self.raiz = NodoABB(estudiante)
        else:
            self._insertar_recursivo(self.raiz, estudiante)

    def _insertar_recursivo(self, nodo, estudiante):
        if estudiante["id"] < nodo.estudiante["id"]:
            if nodo.izq is None:
                nodo.izq = NodoABB(estudiante)
            else:
                self._insertar_recursivo(nodo.izq, estudiante)
        elif estudiante["id"] > nodo.estudiante["id"]:
            if nodo.der is None:
                nodo.der = NodoABB(estudiante)
            else:
                self._insertar_recursivo(nodo.der, estudiante)

    def buscar(self, id_buscado):
        return self._buscar_recursivo(self.raiz, id_buscado)

    def _buscar_recursivo(self, nodo, id_buscado):
        if nodo is None:
            return None
        if nodo.estudiante["id"] == id_buscado:
            return nodo.estudiante
        elif id_buscado < nodo.estudiante["id"]:
            return self._buscar_recursivo(nodo.izq, id_buscado) # Descartamos rama derecha
        else:
            return self._buscar_recursivo(nodo.der, id_buscado) # Descartamos rama izquierda

# ==========================================
# 4. ESTRUCTURA 3: ÁRBOL B+ (Simulación enfocada en búsqueda/índices)
# ==========================================
# Escribir un B+ puro en Python es extenso. Esta es una versión simplificada
# que respeta la lógica de separar índices de las hojas y la búsqueda O(log n).
import bisect

class ArbolBPlusSimulado:
    def __init__(self, orden=4):
        # En un B+ real, tendríamos nodos internos y hojas enlazadas.
        # Aquí usamos diccionarios y listas ordenadas para simular el enrutamiento veloz.
        self.claves = []
        self.datos = {}

    def insertar(self, estudiante):
        # Inserción rápida manteniendo el orden de las hojas
        if estudiante["id"] not in self.datos:
            bisect.insort(self.claves, estudiante["id"])
            self.datos[estudiante["id"]] = estudiante

    def buscar(self, id_buscado):
        # Búsqueda binaria simulando el descenso por los nodos internos
        idx = bisect.bisect_left(self.claves, id_buscado)
        if idx != len(self.claves) and self.claves[idx] == id_buscado:
            return self.datos[self.claves[idx]]
        return None

# ==========================================
# 5. MOTOR DE BENCHMARK (Las Pruebas)
# ==========================================
def ejecutar_benchmark(tipo_insercion):
    print(f"\n{'='*50}")
    print(f"⚙️ INICIANDO PRUEBA CON IDs: {tipo_insercion.upper()}")
    print(f"{'='*50}")

    es_ordenado = (tipo_insercion == "ordenados")
    datos = generar_estudiantes(10000, ordenado=es_ordenado)
    ids_a_buscar = [random.choice(datos)["id"] for _ in range(100)]

    # 1. Preparar estructuras
    lista = BusquedaLista()
    abb = ArbolABB()
    bplus = ArbolBPlusSimulado()

    # 2. Poblar estructuras
    for est in datos:
        lista.insertar(est)
        abb.insertar(est)
        bplus.insertar(est)

    # 3. Medir Tiempos (100 búsquedas)
    # -- LISTA --
    inicio = time.perf_counter()
    for i in ids_a_buscar:
        lista.buscar(i)
    tiempo_lista = time.perf_counter() - inicio

    # -- ABB --
    inicio = time.perf_counter()
    for i in ids_a_buscar:
        abb.buscar(i)
    tiempo_abb = time.perf_counter() - inicio

    # -- B+ --
    inicio = time.perf_counter()
    for i in ids_a_buscar:
        bplus.buscar(i)
    tiempo_bplus = time.perf_counter() - inicio

    # 4. Mostrar Resultados
    print(f"Resultados para 10,000 estudiantes (100 búsquedas):")
    print(f"▶ Lista:      {tiempo_lista:.6f} segundos")
    print(f"▶ Árbol ABB:  {tiempo_abb:.6f} segundos")
    print(f"▶ Árbol B+:   {tiempo_bplus:.6f} segundos")

# Ejecutar ambas pruebas
ejecutar_benchmark("aleatorios")
ejecutar_benchmark("ordenados")


⚙️ INICIANDO PRUEBA CON IDs: ALEATORIOS
Resultados para 10,000 estudiantes (100 búsquedas):
▶ Lista:      0.047165 segundos
▶ Árbol ABB:  0.000598 segundos
▶ Árbol B+:   0.000194 segundos

⚙️ INICIANDO PRUEBA CON IDs: ORDENADOS
Resultados para 10,000 estudiantes (100 búsquedas):
▶ Lista:      0.020482 segundos
▶ Árbol ABB:  0.150890 segundos
▶ Árbol B+:   0.000157 segundos
